In [0]:
%pip install python-calamine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.2/925.2 kB 18.3 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np

FILE_PATH = "/Volumes/workspace/default/walmart_data/walmart_supply_chain_500k_clean11.xlsx"

df = pd.read_excel(FILE_PATH, engine="calamine")

print(f"✅ Dataset reloaded")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")

✅ Dataset reloaded
   Rows    : 500,000
   Columns : 39


# 02 — Data Cleaning

## Goal
Fix data type issues, create useful derived columns, and prepare a
clean version of the dataset for analysis.

## What we will do
1. Fix data types (event_ts, on_time_flag, order_line_id)
2. Create derived columns (margin, is_delayed, year, month)
3. Create a focused analysis-ready dataframe
4. Save the clean dataframe for use in the next notebook

In [0]:
# WHY: Even though event_ts loaded correctly as datetime,
# we want to be explicit and consistent.
# on_time_flag should be 0/1 integer but shows as float because of nulls.
# order_line_id is an ID — it should never have decimals.

# Fix event_ts — ensure it's datetime
df['event_ts'] = pd.to_datetime(df['event_ts'])

# Fix on_time_flag — convert float (1.0, 0.0) to integer, keep nulls as is
# We use 'Int64' (capital I) which supports nulls unlike regular int
df['on_time_flag'] = df['on_time_flag'].astype('Int64')

# Fix order_line_id — it's an ID, should be string not float
df['order_line_id'] = df['order_line_id'].astype('Int64')

print("✅ Data types fixed")
print(f"  event_ts       → {df['event_ts'].dtype}")
print(f"  on_time_flag   → {df['on_time_flag'].dtype}")
print(f"  order_line_id  → {df['order_line_id'].dtype}")

✅ Data types fixed
  event_ts       → datetime64[ns]
  on_time_flag   → Int64
  order_line_id  → Int64


In [0]:
# WHY: Derived columns are new columns we calculate FROM existing ones.
# They make analysis much easier later.
# A good analyst always asks "what extra information can I extract from what I have?"

# 1. MARGIN — how much profit per unit
df['margin'] = df['sell_price'] - df['unit_cost']
df['margin_pct'] = ((df['margin'] / df['sell_price']) * 100).round(2)

# 2. IS_DELAYED — simpler version of delay_days for quick filtering
df['is_delayed'] = (df['delay_days'] > 0).astype('Int64')

# 3. EXTRACT DATE PARTS — useful for trend analysis
df['year'] = df['event_ts'].dt.year
df['month'] = df['event_ts'].dt.month
df['month_name'] = df['event_ts'].dt.strftime('%b')  # Jan, Feb, Mar...
df['quarter'] = df['event_ts'].dt.quarter

# 4. HAS_EXCEPTION — quick flag for whether an event had any exception
df['has_exception'] = df['exception_code'].notna().astype(int)

# Verify all new columns were created
new_cols = ['margin', 'margin_pct', 'is_delayed', 'year', 'month', 'month_name', 'quarter', 'has_exception']
print("✅ Derived columns created:")
for col in new_cols:
    print(f"   {col:<15} sample value: {df[col].iloc[0]}")

✅ Derived columns created:
   margin          sample value: 169.0
   margin_pct      sample value: 30.78
   is_delayed      sample value: 0
   year            sample value: 2025
   month           sample value: 4
   month_name      sample value: Apr
   quarter         sample value: 2
   has_exception   sample value: 1


In [0]:
# WHY: The full dataset has 39 columns, many of which we won't use
# for operational efficiency analysis. 
# Creating a focused dataframe makes our EDA code cleaner and faster.
# We also filter out 2016-2023 here since those years have very low
# event counts and may not be fully recorded.

df_clean = df[df['year'] >= 2024].copy()

print("✅ Analysis-ready dataframe created")
print(f"   Original rows : {len(df):,}")
print(f"   Filtered rows : {len(df_clean):,}")
print(f"   Rows removed  : {len(df) - len(df_clean):,} (2016-2023 low-volume years)")
print(f"   Columns       : {df_clean.shape[1]}")
print(f"\n   Year breakdown:")
print(df_clean['year'].value_counts().sort_index().to_string())

✅ Analysis-ready dataframe created
   Original rows : 500,000
   Filtered rows : 356,048
   Rows removed  : 143,952 (2016-2023 low-volume years)
   Columns       : 47

   Year breakdown:
year
2024    177223
2025    178438
2026       387


In [0]:
# WHY: 2026 only has 387 events (13 days of data).
# Including it would give misleading results if we compare yearly totals.
# A good analyst never compares a full year against a partial year.

df_clean = df_clean[df_clean['year'] != 2026].copy()

print("✅ Removed incomplete year 2026")
print(f"   Final rows : {len(df_clean):,}")
print(f"\n   Year breakdown:")
print(df_clean['year'].value_counts().sort_index().to_string())

✅ Removed incomplete year 2026
   Final rows : 355,661

   Year breakdown:
year
2024    177223
2025    178438


In [0]:
# WHY: Before moving to EDA, always do a final sanity check.
# Confirm your key columns look right and the data is ready.

print("FINAL VALIDATION")
print("=" * 45)
print(f"  Rows            : {len(df_clean):,}")
print(f"  Columns         : {df_clean.shape[1]}")
print(f"  Date range      : {df_clean['event_ts'].min().date()} → {df_clean['event_ts'].max().date()}")
print(f"  Years included  : {sorted(df_clean['year'].unique())}")
print(f"\n  KEY METRICS:")
on_time = df_clean['on_time_flag'].dropna()
print(f"  On-time rate    : {(on_time == 1).sum() / len(on_time) * 100:.1f}%")
print(f"  Anomaly rate    : {df_clean['anomaly_flag'].mean() * 100:.1f}%")
print(f"  Avg delay days  : {df_clean['delay_days'].mean():.2f}")
print(f"  Revenue leakage : {df_clean['revenue_leakage_flag'].mean() * 100:.1f}%")
print(f"\n  NEW COLUMNS ADDED:")
new_cols = ['margin', 'margin_pct', 'is_delayed', 'year', 'month', 'month_name', 'quarter', 'has_exception']
for col in new_cols:
    print(f"  ✅ {col}")

FINAL VALIDATION
  Rows            : 355,661
  Columns         : 47
  Date range      : 2024-01-01 → 2025-12-31
  Years included  : [np.int32(2024), np.int32(2025)]

  KEY METRICS:
  On-time rate    : 70.7%
  Anomaly rate    : 31.2%
  Avg delay days  : 0.75
  Revenue leakage : 9.8%

  NEW COLUMNS ADDED:
  ✅ margin
  ✅ margin_pct
  ✅ is_delayed
  ✅ year
  ✅ month
  ✅ month_name
  ✅ quarter
  ✅ has_exception


In [0]:
# WHY: We need to save df_clean so we can load it directly
# in the next notebook without re-running all the cleaning steps.
# Saving as CSV to our Databricks Volume is the simplest approach.

SAVE_PATH = "/Volumes/workspace/default/walmart_data/walmart_clean.csv"

df_clean.to_csv(SAVE_PATH, index=False)

print(f"✅ Clean dataset saved successfully")
print(f"   Location : {SAVE_PATH}")
print(f"   Rows     : {len(df_clean):,}")
print(f"   Columns  : {df_clean.shape[1]}")

✅ Clean dataset saved successfully
   Location : /Volumes/workspace/default/walmart_data/walmart_clean.csv
   Rows     : 355,661
   Columns  : 47


## ✅ Data Cleaning Complete

### What we did:
1. Fixed data types — event_ts (datetime), on_time_flag (Int64), order_line_id (Int64)
2. Created 8 derived columns — margin, margin_pct, is_delayed, year, month, month_name, quarter, has_exception
3. Filtered to 2024–2025 — removed low-volume years and incomplete 2026
4. Saved clean dataset to Volume for use in EDA notebook

### Clean dataset summary:
- Rows: 355,661
- Columns: 47
- Date range: 2024-01-01 to 2025-12-31
- On-time rate: 70.7%
- Anomaly rate: 31.2%
- Revenue leakage: 9.8%

### Key decision:
We did NOT drop any null rows. Nulls are structural in this dataset —
they only appear where a column doesn't apply to that event type.